# Gear 2.2 follow-up — zero-inflated / hurdle response

**Exploratory.** Не переписывает первый эксперимент. Канон не трогаем.

Статус: нет нового окна после 2026-08-19 12:00Z → confirmatory replication недоступна.

Выход: `research/output/gear22_signal_fill_hurdle/` · note: `research/gear22_signal_fill_hurdle.md`.


In [ ]:
from pathlib import Path
import json
import pandas as pd

OUT = Path("research/output/gear22_signal_fill_hurdle")
meta = json.loads((OUT / "run_meta.json").read_text())
verdict = json.loads((OUT / "verdict.json").read_text())
primary = json.loads((OUT / "primary_1h.json").read_text())
print("status:", meta["status"])
print("VERDICT:", verdict["verdict"])
print("checks:", json.dumps(verdict["checks"], ensure_ascii=False, indent=2))


## Таблица hurdle по |z|-бинам (point + 1h CI)

In [ ]:
rows = []
point = {b["bin"]: b for b in primary["point"]["bins"]}
boot = {b["bin"]: b for b in primary["bootstrap"]}
for name in ["(0,1]", "(1,2]", "(2,4]", ">4"]:
    p, b = point[name], boot[name]
    def fmt(key):
        x = b[key]
        return f"{x['point']:.3f} [{x['ci_low']:.3f}, {x['ci_high']:.3f}]"
    rows.append({
        "bin": name,
        "n": p["n"],
        "p_move": round(p["p_move"], 3),
        "p+_collapse": fmt("p_plus_collapse"),
        "I+": fmt("I_plus"),
        "I-": fmt("I_minus"),
        "A": fmt("A"),
        "med_collapse": round(p["med_collapse_size"], 4),
        "med_continue": round(p["med_continue_size"], 4),
    })
pd.DataFrame(rows)


## Sensitivity 4h / day / equal-coin

In [ ]:
for fname in ["sensitivity_4h.json", "sensitivity_day.json", "equal_coin_1h.json"]:
    res = json.loads((OUT / fname).read_text())
    print("===", fname, "n_blocks=", res.get("n_blocks"), "equal_coin=", res.get("equal_coin"))
    src = res["bootstrap"] if res.get("bootstrap") else res["point"]["bins"]
    if res.get("bootstrap"):
        for b in src:
            if b["bin"] in ("(2,4]", ">4"):
                print(b["bin"], "I+", b["I_plus"], "A", b["A"])
    else:
        for b in src:
            if b["bin"] in ("(2,4]", ">4"):
                print(b["bin"], "I+", b.get("I_plus"), "A", b.get("A"), "p+", b.get("p_plus_collapse"))


## Leg attribution

In [ ]:
legs = json.loads((OUT / "leg_attribution.json").read_text())
pd.DataFrame([
    {"slice": k, "n": v.get("n"), "p_collapse": v.get("p_collapse"),
     "share_okx_mid": v.get("share_okx_mid_dominates"),
     "share_bybit_mid": v.get("share_bybit_mid_dominates")}
    for k, v in legs.items() if isinstance(v, dict)
])


## Continuous curves (не критерий успеха)

In [ ]:
curves = pd.read_csv(OUT / "continuous_curves.csv")
curves


## Stop

Первый эксперимент не переписан. Канон / entry rules / handoff — нет.
